# Expected-GPA Metrics and Nominal Inference

This notebook reproduces the historical budget-only expected-individual-GPA target, allocation-distance metrics, equal-person inference, and the 24 conditional power references. A separate final section maps existing H6/H7 tests to prospective surplus-primary and EGPA-secondary roles without changing historical results. These routes are separate from the historical capped calibration. See the [method settings and scope](README.md#expected-gpa-and-nominal-inference).

The target minimizes `abs(E[GPA_A] - E[GPA_B])` on the 1001-allocation grid, not the expected absolute realized gap. Ties favor higher combined expected GPA, then closeness to 50/50, then the lower grid index. Maximizing combined GPA is a different operation.

`R_EGPA_pp = (abs(j - 500) - abs(j - j_EGPA)) / 10`: positive values mean the allocation is closer to expected-GPA equality than to equal split. It is not a hidden true-outcome equality measure.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from a repository checkout.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])


## Scope

The full target panel has 27 working initial states and three allocation rules per state; it is deterministic target geometry, not an evaluation of a new acquisition policy. The power route reproduces the 24 declared-assumption cases, including the N versus N-1 checks and retention references.

The historical person-level inference roster uses ten main trials per person, nine separate nominal tests at alpha=.05, fixed directional tails, and pointwise 95% intervals. It is preserved for reproduction, not re-adopted as the current complete study analysis. It makes no familywise guarantee. Missing arms, zero standard errors and unsupported projections remain unavailable; supporting analyses cannot rescue a primary test.

The raw-trial input route is synthetic only. It recomputes beliefs and targets from the supplied histories, rather than accepting cached target columns. It is not a validated human-session importer. The complete 400,000-attempt calibration and independent truth tables are reproduced in [native_state_calibration.ipynb](native_state_calibration.ipynb). Neither notebook establishes latest-model cap20 compatibility.


In [ ]:
import json
from IPython.display import display

RUN = False
MODE = 'smoke'  # 'full' produces all 27 initial target states.
OUTPUT = ROOT / 'results' / 'expected_gpa_reproduction'
RAW_TRIALS = None  # Optional Path to a synthetic raw-trial JSON array.
if MODE not in {'smoke', 'full'}:
    raise ValueError('MODE must be smoke or full')
runner = ROOT / 'scripts' / 'reproduce_expected_gpa.py'
commands = [
    [sys.executable, str(runner), 'targets', '--output', str(OUTPUT / 'targets')],
    [sys.executable, str(runner), 'power', '--output', str(OUTPUT / 'power')],
]
if MODE == 'smoke':
    commands[0] += ['--template', 'C3_s0_g0']
if RAW_TRIALS is not None:
    commands.append([sys.executable, str(runner), 'analyze', '--trials', str(Path(RAW_TRIALS)),
                     '--output', str(OUTPUT / 'analysis')])
for command in commands:
    print(' '.join(command))
    if RUN:
        subprocess.check_call(command, cwd=ROOT)


## Synthetic Trial Input

`analyze` takes a JSON array. Each record has `templateId`, `participantId`, `mainIndex` (1..10), `paid` (ordered `{student: 0|1, report: 1..100}` objects), `shareIndex` (0..1000, or null if no allocation), and `humanObservations: false`. Optional fields are `sessionId`, `sideReversed`, `recordPresent`, and `endpointPresent`.

Template identities come from `scripts.reproduce_expected_gpa.templates()`. The working C1 total centre gap is one minute. Remaining budget and posteriors are reconstructed; cached expected-GPA targets are not inputs. Complete people have exactly ten distinct main indices in one condition/session. Partial records remain explicitly ineligible under the corresponding estimand.


In [ ]:
if RUN:
    targets = json.loads((OUTPUT / 'targets' / 'result.json').read_text())
    power = json.loads((OUTPUT / 'power' / 'result.json').read_text())
    display({'target_states': targets['templates'], 'target_rows': len(targets['records']),
             'power_cases': len(power['rows']), 'final_enrollment_selected': power['finalNSelected']})
    display(targets['records'][:3])
    display([r for r in power['rows'] if r['d'] == .3 and r['targetPower'] == .8
             and r['tail'] == 'one_sided_primary'])
    if RAW_TRIALS is not None:
        analysis = json.loads((OUTPUT / 'analysis' / 'result.json').read_text())
        display(analysis['tests'])
else:
    print('No calculations or result reads; set RUN=True after checking settings.')


## Prospective H6/H7: Surplus Primary, EGPA Secondary

The prospective primary is resemblance to equal posterior-mean surplus at the actual terminal posterior and remaining positive budget. For exact posterior means `mA,mB` and budget `B`, the continuous target is `clip(0.5 + (mA-mB)/(2*B), 0, 1)`, projected using the existing native surplus rule. The paired trial endpoint is `(abs(j-500)-abs(j-jSurplus))/10`; positive values indicate closer allocation to surplus equality than to equal division. The existing source averages ten ordered trials per eligible person, with equal person weights. Display-rounded means, expected-GPA tie rules and available-history substitutions are not used.

[h6_h7_surplus.json](../../configs/h6_h7_surplus.json) specifies only H6/H7. It maps historical `S_surplus_C2/C3` to prospective `H06/H07_R_SURPLUS_C2/C3`, and historical `H06/H07_R_EGPA_C2/C3` to secondary `S_R_EGPA_C2/C3`. Greater-tail one-sample Student tests remain nominal alpha=.05 with pointwise two-sided 95% intervals. Every embedded historical result keeps its ID, primary flag, namespace, counts, numerical values and unavailable reason. A secondary result cannot rescue the primary. Other hypotheses and the historical configuration are unchanged.

The default `historical_fixture` route reconstructs the original deterministic **25-person/250-row algebra fixture**, with five people in each of C1-C5 and ten ordered trials per person. [h6_h7_algebra_fixture.json](../../configs/h6_h7_algebra_fixture.json) fixes the complete recipe and reference numerical environment (Python 3.10.9, NumPy 1.23.5, SciPy 1.10.0). For zero-based arm/person indices `a,p` and main index `m=1..10`, allocation is `500+(m-5)*(p+1)*(3 if a%2==0 else -3)+2*p`; expected-GPA target is `500+((m+a)%5-2)*7` and surplus target is three indices higher. Remaining budget uses the prescribed `(0,1,3)[(m+p)%3]` paid counts. Exact rational centered-OLS weights are rounded once to binary64 and retained for the initial projections. No seed or stochastic response generator is used.

These are **non-engine algebra API inputs**, not native task histories or model-generated surplus targets. Replacing their prescribed target indices with reconstructed native targets would not reproduce the accepted fixture. The `fixture` stage feeds these recovered inputs into the existing public `allocation_metrics` and `nominal_inference.analyze` functions, writing metrics, the original-role analysis, recipe and completion metadata. The following map/readback stages preserve each generated historical test unchanged. Public schema names and fresh provenance are deliberately distinct from the original archive; no historical writer, acceptance receipt or whole-file byte identity is impersonated. Exact same-environment numerical parity remains subject to execution verification; cross-version regression tolerance is not bitwise identity.

Set `SURPLUS_INPUT='illustrative'` only for the separate four-person/40-trial native-history wiring example, which does **not** reproduce the historical fixture numbers. Alternatively, set `SURPLUS_ANALYSIS` to an existing public nominal result to map it without recalculation. Private archived-result schemas are not silently relabeled as public-generated inputs.

The independent checker verifies the four exact historical payloads, role mapping, population counts and content identity without importing the mapper or recalculating statistics. It does not authenticate scientific acceptance or establish human-data admission, observed direction, calibration, power, a causal mechanism or global error control. The earlier expected-GPA calibration/power routes do not automatically validate surplus inference. All calculations below are disabled by default.


In [ ]:
import shlex

RUN_SURPLUS = False
SURPLUS_INPUT = 'historical_fixture'  # Optional 'illustrative' native-history example.
SURPLUS_ANALYSIS = None  # Optional existing PUBLIC nominal result.json; no recalculation.
SURPLUS_OUTPUT = ROOT / 'results' / 'h6_h7_surplus_reproduction'
SURPLUS_CONFIG = ROOT / 'configs' / 'h6_h7_surplus.json'
FIXTURE_CONFIG = ROOT / 'configs' / 'h6_h7_algebra_fixture.json'
if SURPLUS_INPUT not in {'historical_fixture', 'illustrative'}:
    raise ValueError('Unknown surplus input route')
mapper = ROOT / 'scripts' / 'map_h6_h7_surplus.py'
checker = ROOT / 'scripts' / 'read_h6_h7_surplus_mapping.py'
surplus_commands = []
if SURPLUS_ANALYSIS is None:
    historical_output = SURPLUS_OUTPUT / 'historical_analysis'
    surplus_analysis = historical_output / 'result.json'
    if SURPLUS_INPUT == 'historical_fixture':
        surplus_commands.append([sys.executable, str(mapper), 'fixture',
                                 '--fixture-config', str(FIXTURE_CONFIG), '--output', str(historical_output)])
    else:
        raw = SURPLUS_OUTPUT / 'synthetic_trials.json'
        surplus_commands.extend([
            [sys.executable, str(mapper), 'example', '--output', str(raw)],
            [sys.executable, str(ROOT / 'scripts' / 'reproduce_expected_gpa.py'), 'analyze',
             '--trials', str(raw), '--output', str(historical_output)],
        ])
else:
    surplus_analysis = Path(SURPLUS_ANALYSIS)
mapping_file = SURPLUS_OUTPUT / 'mapping.json'
check_file = SURPLUS_OUTPUT / 'independent.json'
surplus_commands.extend([
    [sys.executable, str(mapper), 'map', '--analysis', str(surplus_analysis),
     '--config', str(SURPLUS_CONFIG), '--output', str(mapping_file)],
    [sys.executable, str(checker), '--analysis', str(surplus_analysis),
     '--config', str(SURPLUS_CONFIG), '--mapping', str(mapping_file), '--output', str(check_file)],
])
for command in surplus_commands:
    print(shlex.join(command))
    if RUN_SURPLUS:
        subprocess.check_call(command, cwd=ROOT)
if RUN_SURPLUS:
    mapped = json.loads(mapping_file.read_text())
    display([{'claim': row['prospectiveClaimId'], 'role': row['prospectiveRole'],
              'historical_id': row['historicalTestId'], 'historical_result': row['historicalResult']}
             for row in mapped['mappings']])
    display(json.loads(check_file.read_text()))
else:
    print('No surplus inputs generated or results read. Outputs must be new; no files are overwritten.')
